# Students’ Mental Health Analysis

A privacy-aware analysis of student wellbeing using a relational model and reusable SQL KPI view.

> **Data note:** This portfolio reconstruction uses a reproducible synthetic dataset because the original training files were not available for publication.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
ROOT=Path("..").resolve()
conn=sqlite3.connect(ROOT/"data/processed/student_wellbeing.db")

## Reusable student KPI view

In [ ]:
query="""SELECT programme, ROUND(AVG(avg_wellbeing),1) avg_wellbeing, ROUND(AVG(avg_stress),1) avg_stress, COUNT(*) students FROM vw_student_kpi GROUP BY programme ORDER BY avg_wellbeing DESC"""
pd.read_sql_query(query,conn)

## CTE cohort analysis

In [ ]:
query="""WITH cohorts AS (SELECT *, CASE WHEN length_of_stay_months<=6 THEN '0-6 months' WHEN length_of_stay_months<=12 THEN '7-12 months' WHEN length_of_stay_months<=24 THEN '13-24 months' ELSE '25+ months' END stay_group FROM vw_student_kpi) SELECT stay_group, ROUND(AVG(avg_wellbeing),1) avg_wellbeing, COUNT(*) students FROM cohorts GROUP BY stay_group"""
pd.read_sql_query(query,conn)

## Privacy check

In [ ]:
cols=pd.read_sql_query("SELECT * FROM vw_student_kpi LIMIT 1",conn).columns.tolist()
print(cols)
assert not {"name","email","address","phone"}.intersection(cols)
conn.close()